In [7]:
import geopandas as gpd
import pyogrio
import folium
from pathlib import Path

DATA      = Path("data")
FRD_PATH  = DATA / "FRD_Coastal_34001.gdb"          # symlink → FRD_Coastal_34001_Geodatabase_20170524
NFHL_PATH = DATA / "NFHL_34_20250903/NFHL_34_20250903.gdb"

In [8]:
# Inventory both GDBs — geometry type + row count per layer
def list_gdb(path, label):
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    for row in gpd.list_layers(path).itertuples():
        try:
            info = pyogrio.read_info(str(path), layer=row.name)
            geom = row.geometry_type or "Table"
            print(f"  {row.name:<45} {geom:<20} {info['features']:>8,} rows")
        except Exception as e:
            print(f"  {row.name:<45} (error: {e})")

list_gdb(FRD_PATH,  "FRD Coastal 34001  (Atlantic County, 2017)")
list_gdb(NFHL_PATH, "NFHL 34  (NJ statewide, 2025)")


  FRD Coastal 34001  (Atlantic County, 2017)
  L_AOMI_Summary                                Table                      48 rows
  L_Claims                                      Table                      21 rows
  L_CSLF_Summary                                Table                       0 rows
  L_Exposure                                    Table                      21 rows
  L_Local_GBS                                   Table                       0 rows
  L_RA_AAL                                      Table                  15,864 rows
  L_RA_Refined                                  Table                   2,697 rows
  L_RA_Summary                                  Table                     105 rows
  L_RA_UDF_Refined                              Table                       0 rows
  L_Source_Cit                                  Table                      43 rows
  FRR_Project                                   Table                       0 rows
  FRR_Images                             

In [9]:
# Load key spatial layers
# NFHL is statewide — clip to FRD project bbox to avoid loading 57k rows unnecessarily
frd_proj = gpd.read_file(FRD_PATH, layer="S_FRD_Proj_Ar", engine="pyogrio").to_crs(4326)
bbox     = tuple(frd_proj.total_bounds)   # (minx, miny, maxx, maxy) in WGS84

nfhl_fhz = gpd.read_file(NFHL_PATH, layer="S_FLD_HAZ_AR", engine="pyogrio", bbox=bbox).to_crs(4326)
nfhl_bfe = gpd.read_file(NFHL_PATH, layer="S_BFE",        engine="pyogrio", bbox=bbox).to_crs(4326)
nfhl_wtr = gpd.read_file(NFHL_PATH, layer="S_WTR_AR",     engine="pyogrio", bbox=bbox).to_crs(4326)
frd_cen  = gpd.read_file(FRD_PATH,  layer="S_CenBlk_Ar",  engine="pyogrio").to_crs(4326)
frd_huc  = gpd.read_file(FRD_PATH,  layer="S_HUC_Ar",     engine="pyogrio").to_crs(4326)

for name, gdf in [("NFHL S_FLD_HAZ_AR", nfhl_fhz), ("NFHL S_BFE", nfhl_bfe),
                  ("NFHL S_WTR_AR", nfhl_wtr), ("FRD S_FRD_Proj_Ar", frd_proj),
                  ("FRD S_CenBlk_Ar", frd_cen), ("FRD S_HUC_Ar", frd_huc)]:
    print(f"  {name:<25} {len(gdf):>6,} rows  {gdf.shape[1]} cols")

  NFHL S_FLD_HAZ_AR          2,668 rows  23 cols
  NFHL S_BFE                   412 rows  10 cols
  NFHL S_WTR_AR                240 rows  11 cols
  FRD S_FRD_Proj_Ar              1 rows  17 cols
  FRD S_CenBlk_Ar            4,688 rows  17 cols
  FRD S_HUC_Ar                 123 rows  9 cols


In [10]:
print("=== NFHL S_FLD_HAZ_AR ===")
print(nfhl_fhz.dtypes.to_string())
print(f"\nFLD_ZONE distribution:\n{nfhl_fhz['FLD_ZONE'].value_counts().to_string()}")

print("\n=== FRD S_CenBlk_Ar ===")
print(frd_cen.dtypes.to_string())

=== NFHL S_FLD_HAZ_AR ===
DFIRM_ID          object
VERSION_ID        object
FLD_AR_ID         object
STUDY_TYP         object
FLD_ZONE          object
ZONE_SUBTY        object
SFHA_TF           object
STATIC_BFE       float64
V_DATUM           object
DEPTH            float64
LEN_UNIT          object
VELOCITY         float64
VEL_UNIT          object
AR_REVERT         object
AR_SUBTRV         object
BFE_REVERT       float64
DEP_REVERT       float64
DUAL_ZONE         object
SOURCE_CIT        object
GFID              object
SHAPE_Length     float64
SHAPE_Area       float64
geometry        geometry

FLD_ZONE distribution:
FLD_ZONE
X                    2012
AE                    452
A                     113
VE                     86
AREA NOT INCLUDED       3
AO                      1
OPEN WATER              1

=== FRD S_CenBlk_Ar ===
CEN_BLK_ID        object
POPULATION         int32
ARV_BG_TOT         int32
ARV_CN_TOT         int32
ARV_BG_RES         int32
ARV_CN_RES         int32
ARV_BG_CO

In [ ]:
ZONE_COLORS = {"A":"#d73027","AE":"#fc8d59","AH":"#fee090","AO":"#ffffbf",
               "VE":"#a50026","V":"#b2182b","X":"#4dac26","OPEN WATER":"#74add1"}

cx, cy = frd_proj.geometry.union_all().centroid.coords[0]
m = folium.Map(location=[cy, cx], zoom_start=11, tiles="CartoDB positron", control_scale=True)

# NFHL — Flood Hazard Zones (colour-coded by zone)
folium.GeoJson(
    nfhl_fhz[["FLD_ZONE","ZONE_SUBTY","SFHA_TF","STATIC_BFE","geometry"]].__geo_interface__,
    name="NFHL — Flood Hazard Zones",
    style_function=lambda f: {
        "fillColor": ZONE_COLORS.get(f["properties"].get("FLD_ZONE","X"), "#cccccc"),
        "color": "none", "fillOpacity": 0.55, "weight": 0},
    tooltip=folium.GeoJsonTooltip(["FLD_ZONE","ZONE_SUBTY","SFHA_TF","STATIC_BFE"]),
    show=True,
).add_to(m)

# NFHL — Base Flood Elevation lines
folium.GeoJson(
    nfhl_bfe[["ELEV","LEN_UNIT","V_DATUM","geometry"]].__geo_interface__,
    name="NFHL — Base Flood Elevations",
    style_function=lambda f: {"color":"#0571b0","weight":1.2,"opacity":0.7},
    tooltip=folium.GeoJsonTooltip(["ELEV","LEN_UNIT","V_DATUM"]),
    show=False,
).add_to(m)

# NFHL — Water areas
folium.GeoJson(
    nfhl_wtr[["geometry"]].__geo_interface__,
    name="NFHL — Water Areas",
    style_function=lambda f: {"fillColor":"#74add1","color":"#4393c3","fillOpacity":0.5,"weight":0.5},
    show=False,
).add_to(m)

# FRD — Census block risk (Average Annualised Risk)
frd_cen_cols = ["CEN_BLK_ID","POPULATION","ARV_BG_TOT","ARV_CN_TOT","ARV_BG_RES","ARV_CN_RES","geometry"]
folium.GeoJson(
    frd_cen[frd_cen_cols].__geo_interface__,
    name="FRD — Census Block Risk (AAR)",
    style_function=lambda f: {"fillColor":"#f46d43","color":"#d73027","fillOpacity":0.25,"weight":0.8},
    tooltip=folium.GeoJsonTooltip(["CEN_BLK_ID","POPULATION","ARV_BG_TOT","ARV_CN_TOT","ARV_BG_RES","ARV_CN_RES"]),
    show=False,
).add_to(m)

# FRD — HUC watersheds
folium.GeoJson(
    frd_huc[["geometry"]].__geo_interface__,
    name="FRD — HUC Watersheds",
    style_function=lambda f: {"fillColor":"none","color":"#5e4fa2","weight":1.2,"dashArray":"4 3","fillOpacity":0},
    show=False,
).add_to(m)

# FRD — Project boundary
folium.GeoJson(
    frd_proj[["geometry"]].__geo_interface__,
    name="FRD — Project Boundary",
    style_function=lambda f: {"fillColor":"none","color":"#7b3294","weight":2.5,"dashArray":"8 4"},
    show=True,
).add_to(m)

legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:1000;background:white;
     padding:10px 14px;border-radius:6px;border:1px solid #ccc;font-size:12px;line-height:1.7">
<b>NFHL Flood Zones</b><br>
<span style="background:#a50026;padding:0 8px">&nbsp;</span> VE — coastal 1%<br>
<span style="background:#d73027;padding:0 8px">&nbsp;</span> A — 1% annual<br>
<span style="background:#fc8d59;padding:0 8px">&nbsp;</span> AE<br>
<span style="background:#ffffbf;padding:0 8px">&nbsp;</span> AO / AH<br>
<span style="background:#4dac26;padding:0 8px">&nbsp;</span> X — minimal risk<br>
<span style="background:#74add1;padding:0 8px">&nbsp;</span> Open Water
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))
folium.LayerControl(collapsed=False).add_to(m)
m